[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsobaCloud/odse/blob/main/colab/durban_feeder_grid_colab_demo.ipynb)

# Durban Feeder Grid Modeling & BESS Dispatch Demo — ODS-E + pvlib

This interactive tutorial demonstrates how to use the **Open Data Schema for Energy (ODS-E)** and its **`odse.adapters.pvlib`** adapter to ingest, standardize, and model multi-site distribution feeder load flows and Battery Energy Storage System (BESS) dispatch.

### Real-World Distribution Feeder Scenario
We model an 11 kV municipal distribution feeder in Durban, KwaZulu-Natal (eThekwini Electricity) supplying three commercial & industrial (C&I) facilities with **5.55 MWp total solar PV** capacity:
1. **Thokoza Logistics Park**: 1.2 MW Solar PV + 1.0 MW / 2.0 MWh BESS (LFP)
2. **Khanyisa Manufacturing Hub**: 3.6 MW Solar PV + 2.0 MW / 4.0 MWh BESS (LFP)
3. **Simunye Cold Storage**: 750 kW Solar PV (rooftop array, no storage)

### Technical Objectives
- **Multi-Site ODS-E Ingestion**: Ingest 6 months of 5-min telemetry and validate schema compliance via `odse.validator.validate_batch`.
- **`odse.adapters.pvlib` Interoperability**: Standardize ODS-E telemetry into `pvlib`-compliant DataFrames with UTC DatetimeIndex and physical variable mappings (`kW` → `p_mp`, `kWh` → `ac_energy`, `poa_irradiance` → `poa_global`).
- **Clear-Sky Physical Yield Modeling**: Compute theoretical clear-sky irradiance via `pvlib.location.Location` (Ineichen model) and benchmark actual inverter performance against physics expectations.
- **Feeder Load Flow & Reverse Power Flow ("Duck Curve")**: Aggregate feeder net load, detect dangerous midday backfeed events (peaking at **-1.66 MW** into the substation transformer), and analyze thermal risks.
- **BESS Coordinated Dispatch**: Absorb midday solar surplus into the 3.0 MW / 6.0 MWh battery fleet and discharge during the 17:00–20:30 evening demand peak.
- **Economic TOU Arbitrage**: Quantify cost avoidance under Eskom Megaflex Time-of-Use tariffs.
- **Roundtrip Interchange**: Export modeled dispatch results back into canonical ODS-E records using `pvlib_to_odse()`.


In [ ]:
!pip install -q odse pandas matplotlib pvlib


In [ ]:
import io
import json
import warnings
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pvlib
from odse.adapters.pvlib import odse_to_pvlib, pvlib_to_odse
from odse.validator import validate_batch

warnings.filterwarnings('ignore')

# Styling palette matching ODS-E guidelines
BLUE = '#455BF1'
AMBER = '#f59e0b'
RED = '#E20419'
GREEN = '#10b981'
PURPLE = '#8b5cf6'
DARK = '#0f172a'
GRAY = '#64748b'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8fafc',
    'axes.edgecolor': '#cbd5e1',
    'axes.grid': True,
    'grid.color': '#e2e8f0',
    'font.family': 'sans-serif',
    'figure.autolayout': True
})

# GitHub raw base URL for kit datasets
BASE = "https://raw.githubusercontent.com/AsobaCloud/odse/main/colab/data"
print("ODS-E + pvlib environment successfully configured.")


## 1. Multi-Site Telemetry Ingestion & ODS-E Validation

We begin by loading 6 months of 5-minute telemetry from our three Durban C&I solar facilities. Every inverter streams standardized ODS-E telemetry including AC power, temperature, operating voltage, MPPT tracking power, irradiance, wind speed, and state codes.


In [ ]:
sites = {
    'thokoza': {'name': 'Thokoza Logistics Park', 'pv_mw': 1.2, 'bess_mw': 1.0, 'bess_mwh': 2.0},
    'khanyisa': {'name': 'Khanyisa Manufacturing Hub', 'pv_mw': 3.6, 'bess_mw': 2.0, 'bess_mwh': 4.0},
    'simunye': {'name': 'Simunye Cold Storage', 'pv_mw': 0.75, 'bess_mw': 0.0, 'bess_mwh': 0.0}
}

pv_dfs = {}
for site_id, info in sites.items():
    url = f"{BASE}/{site_id}_telemetry_6mo.csv.gz"
    # Fallback to local file if running locally in odse repo
    try:
        df = pd.read_csv(url, parse_dates=['timestamp'])
    except Exception:
        df = pd.read_csv(f"colab/data/{site_id}_telemetry_6mo.csv.gz", parse_dates=['timestamp'])
    pv_dfs[site_id] = df
    print(f"Loaded {info['name']}: {len(df):,} inverter records ({df['asset_id'].nunique()} inverters)")

# Validate sample batch against ODS-E schema
sample_df = pv_dfs['thokoza'].head(50)
sample_records = []
for r in sample_df.itertuples():
    sample_records.append({
        'timestamp': r.timestamp.isoformat() if hasattr(r.timestamp, 'isoformat') else str(r.timestamp),
        'kWh': float(r.kwh),
        'kW': float(r.power),
        'voltage_ac': float(r.voltage),
        'temperature': float(r.temperature),
        'poa_irradiance': float(r.irradiance),
        'wind_speed_ms': float(r.wind_speed),
        'error_type': str(r.error_type),
        'asset_id': str(r.asset_id),
    })

validation_result = validate_batch(sample_records)
is_valid = getattr(validation_result, 'is_valid', validation_result.invalid == 0)
status_str = 'PASSED (Valid)' if is_valid else 'FAILED'
print(f"\nODS-E Schema Validation: {status_str}")
print(f"Validated {validation_result.valid}/{validation_result.total} telemetry records against canonical ODS-E specification.")
assert is_valid, 'ODS-E schema validation failed'


## 2. Ingesting Feeder Grid Load & BESS Telemetry

Next, we load the simultaneous facility consumption (building load) and BESS operation logs for each site. This gives us the full time-series profile of generation, demand, and storage across the 11 kV feeder.


In [ ]:
grid_dfs = {}
bess_dfs = {}

for site_id in sites.keys():
    try:
        g_df = pd.read_csv(f"{BASE}/{site_id}_grid_load_6mo.csv.gz", parse_dates=['timestamp'])
    except Exception:
        g_df = pd.read_csv(f"colab/data/{site_id}_grid_load_6mo.csv.gz", parse_dates=['timestamp'])
    grid_dfs[site_id] = g_df
    
    if sites[site_id]['bess_mw'] > 0:
        try:
            b_df = pd.read_csv(f"{BASE}/{site_id}_bess_6mo.csv.gz", parse_dates=['timestamp'])
        except Exception:
            b_df = pd.read_csv(f"colab/data/{site_id}_bess_6mo.csv.gz", parse_dates=['timestamp'])
        bess_dfs[site_id] = b_df

# Summary Table
summary_data = []
for site_id, info in sites.items():
    g = grid_dfs[site_id]
    summary_data.append({
        'Site': info['name'],
        'PV Capacity (MWp)': info['pv_mw'],
        'BESS Rating': f"{info['bess_mw']} MW / {info['bess_mwh']} MWh" if info['bess_mw'] > 0 else 'None',
        'Total Generation (MWh)': g['solar_generation_kw'].sum() * (5/60) / 1000,
        'Total Building Demand (MWh)': g['building_load_kw'].sum() * (5/60) / 1000,
        'Peak Solar Output (kW)': g['solar_generation_kw'].max(),
        'Peak Load (kW)': g['building_load_kw'].max()
    })

summary_df = pd.DataFrame(summary_data).set_index('Site')
display(summary_df.round(1))


## 3. Standardizing with `odse_to_pvlib` Adapter & Clear-Sky Modeling

The `odse.adapters.pvlib` adapter bridges ODS-E schema payloads directly into `pvlib` DataFrames. We convert the Thokoza inverter telemetry, map canonical electrical variables (`kW` $\rightarrow$ `p_mp`, `poa_irradiance` $\rightarrow$ `poa_global`), and compare actual array yield against a theoretical **Ineichen Clear-Sky model** for Durban (`-29.8587° S, 31.0218° E`).


In [ ]:
# Convert sample site telemetry to pvlib DataFrame via ODS-E adapter
thokoza_pv = pv_dfs['thokoza']
sample_inverter = thokoza_pv[thokoza_pv['asset_id'] == 'INV-THK-01'].copy()
# Rename to canonical ODS-E keys for mapping
sample_inverter = sample_inverter.rename(columns={'power': 'kW', 'kwh': 'kWh', 'irradiance': 'poa_irradiance'})

payload = {
    'metadata': {'site_id': 'Thokoza', 'asset_id': 'INV-THK-01', 'rated_kw': 200},
    'telemetry': sample_inverter.head(288).to_dict(orient='records') # 24 hours (288 5-min intervals)
}

pvd, meta = odse_to_pvlib(payload, map_variables=True)
print(f"pvlib DataFrame constructed: {pvd.shape[0]} rows, index type: {type(pvd.index)}")
print(f"Mapped columns: {list(pvd.columns[:6])}...")

# Model Durban clear sky irradiance using pvlib
durban_loc = pvlib.location.Location(
    latitude=-29.8587,
    longitude=31.0218,
    tz='Africa/Johannesburg',
    altitude=10,
    name='Durban Harbour'
)

# Align timestamps
times = pvd.index.tz_convert('Africa/Johannesburg')
cs = durban_loc.get_clearsky(times)
solpos = durban_loc.get_solarposition(times)

# Calculate POA Clear-Sky Irradiance (20-degree tilt facing North)
poa_cs = pvlib.irradiance.get_total_irradiance(
    surface_tilt=20,
    surface_azimuth=0,
    solar_zenith=solpos['zenith'],
    solar_azimuth=solpos['azimuth'],
    dni=cs['dni'],
    ghi=cs['ghi'],
    dhi=cs['dhi']
)

# Plot comparison: Actual Output vs Clear-Sky Irradiance
fig, ax1 = plt.subplots(figsize=(12, 5))
color_actual = BLUE
color_cs = AMBER

ax1.set_xlabel('Time of Day (SAST)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Inverter AC Power (kW)', color=color_actual, fontsize=11, fontweight='bold')
ax1.plot(times, pvd['p_mp'], color=color_actual, lw=2.5, label='Actual Inverter Power (ODS-E)')
ax1.tick_params(axis='y', labelcolor=color_actual)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

ax2 = ax1.twinx()
ax2.set_ylabel('Clear-Sky POA Irradiance (W/m²)', color=color_cs, fontsize=11, fontweight='bold')
ax2.plot(times, poa_cs['poa_global'], color=color_cs, lw=2, linestyle='--', label='pvlib Ineichen Clear-Sky POA')
ax2.tick_params(axis='y', labelcolor=color_cs)
ax2.grid(False)

plt.title('Durban Site: ODS-E Measured Power vs pvlib Clear-Sky Irradiance Model', fontsize=13, fontweight='bold', pad=12)
fig.tight_layout()
plt.show()


## 4. Feeder Load Flow & Reverse Power Flow ("Duck Curve") Analysis

When multiple C&I sites inject peak midday solar into an 11 kV distribution feeder, generation often overtakes local demand. 

Without energy storage, the unmitigated net grid power:
$$\text{Feeder Net Power (Unmitigated)} = \sum \text{Building Load} - \sum \text{Solar Generation}$$

When Net Power becomes **negative**, current flows in reverse through the step-down distribution transformer back onto the municipal grid. This causes voltage rise, tripping protection relays, and violating utility interconnect agreements.


In [ ]:
# Aggregate total feeder profiles
t_g = grid_dfs['thokoza']
k_g = grid_dfs['khanyisa']
s_g = grid_dfs['simunye']

timestamps = t_g['timestamp']
total_solar = t_g['solar_generation_kw'] + k_g['solar_generation_kw'] + s_g['solar_generation_kw']
total_load = t_g['building_load_kw'] + k_g['building_load_kw'] + s_g['building_load_kw']
total_bess_power = t_g['bess_power_kw'] + k_g['bess_power_kw'] # Negative = charging, Positive = discharging

feeder_net_unmitigated = total_load - total_solar
feeder_net_mitigated = total_load - total_solar - total_bess_power

# Compute Reverse Power Flow metrics
unmitigated_backfeed = feeder_net_unmitigated[feeder_net_unmitigated < 0]
mitigated_backfeed = feeder_net_mitigated[feeder_net_mitigated < 0]

print("=== 11 kV FEEDER POWER FLOW AUDIT (6-MONTH HORIZON) ===")
print(f"Feeder Peak Solar Output:     {total_solar.max():,.1f} kW ({total_solar.max()/1000:.2f} MW)")
print(f"Feeder Peak Facility Demand:  {total_load.max():,.1f} kW ({total_load.max()/1000:.2f} MW)")
print(f"Maximum Unmitigated Backfeed: {unmitigated_backfeed.min():,.1f} kW (Reverse power flow)")
print(f"Unmitigated Backfeed Hours:   {len(unmitigated_backfeed) * (5/60):,.1f} hours")
print(f"Maximum Mitigated Backfeed:   {mitigated_backfeed.min():,.1f} kW")
print(f"Mitigated Backfeed Hours:     {len(mitigated_backfeed) * (5/60):,.1f} hours ({100*(1 - len(mitigated_backfeed)/len(unmitigated_backfeed)):.1f}% reduction)")

# Plot 3-Day Feeder Duck Curve
plot_mask = (timestamps >= '2026-03-22') & (timestamps <= '2026-03-25')
p_times = timestamps[plot_mask]
p_solar = total_solar[plot_mask]
p_load = total_load[plot_mask]
p_net = feeder_net_unmitigated[plot_mask]

plt.figure(figsize=(14, 6))
plt.plot(p_times, p_load, color=DARK, lw=2, label='Total Feeder Demand (kW)')
plt.plot(p_times, p_solar, color=AMBER, lw=2, label='Total Solar PV Generation (kW)')
plt.plot(p_times, p_net, color=RED, lw=2.5, label='Net Feeder Power at Substation (kW)')
plt.axhline(0, color='black', linestyle=':', lw=1.5, alpha=0.8)

# Highlight reverse power flow zone
plt.fill_between(p_times, p_net, 0, where=(p_net < 0), color=RED, alpha=0.25, label='Reverse Power Flow (Backfeed Zone)')

plt.title('Durban 11 kV Feeder: Unmitigated Solar Penetration & Transformer Backfeed', fontsize=14, fontweight='bold', pad=12)
plt.ylabel('Power (kW)', fontsize=11, fontweight='bold')
plt.xlabel('Date / Time (UTC)', fontsize=11, fontweight='bold')
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %d %H:%M'))
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()


## 5. BESS Coordinated Dispatch & Backfeed Mitigation

To protect the distribution transformer from backfeed while maximizing financial returns, the BESS fleet operates in a dual-stage dispatch profile:
1. **Solar Absorption Mode (10:00–14:30)**: When solar generation exceeds facility load, BESS units at Thokoza and Khanyisa charge at up to -3,000 kW combined, absorbing excess energy into battery storage.
2. **Peak Shaving Mode (17:00–20:30)**: As the evening Eskom Megaflex Peak tariff window arrives, the batteries discharge stored solar energy to offset expensive grid power.


In [ ]:
# Plot 3-Day Mitigated Power Flow & BESS Behavior
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Panel 1: Feeder Net Load Before vs After BESS
ax1.plot(p_times, feeder_net_unmitigated[plot_mask], color=RED, lw=1.8, linestyle='--', label='Unmitigated Net Power (No BESS)')
ax1.plot(p_times, feeder_net_mitigated[plot_mask], color=BLUE, lw=2.5, label='Mitigated Net Power (With BESS Coordinated Dispatch)')
ax1.axhline(0, color='black', linestyle=':', lw=1.2, alpha=0.7)
ax1.fill_between(p_times, feeder_net_unmitigated[plot_mask], 0, where=(feeder_net_unmitigated[plot_mask] < 0), color=RED, alpha=0.2, label='Unmitigated Backfeed Risk')
ax1.set_ylabel('Net Substation Power (kW)', fontsize=11, fontweight='bold')
ax1.set_title('BESS Coordinated Dispatch: Reverse Power Flow Elimination & Peak Shaving', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right')

# Panel 2: BESS Power & State of Charge
b_power_p = total_bess_power[plot_mask]
soc_thokoza = t_g['bess_soc_pct'][plot_mask]
soc_khanyisa = k_g['bess_soc_pct'][plot_mask]
avg_soc = (soc_thokoza * 2.0 + soc_khanyisa * 4.0) / 6.0 # Weighted average SoC

ax2.fill_between(p_times, b_power_p, 0, where=(b_power_p < 0), color=GREEN, alpha=0.4, label='BESS Charging (Absorbing Midday Solar)')
ax2.fill_between(p_times, b_power_p, 0, where=(b_power_p > 0), color=PURPLE, alpha=0.4, label='BESS Discharging (Evening Peak Shaving)')
ax2.plot(p_times, b_power_p, color=DARK, lw=1.5, label='Total BESS Power (kW)')
ax2.set_ylabel('BESS Power (kW)', fontsize=11, fontweight='bold')

# Secondary axis for Battery SoC %
ax2_twin = ax2.twinx()
ax2_twin.plot(p_times, avg_soc, color='#d97706', lw=2.2, linestyle='-.', label='Fleet Average SoC (%)')
ax2_twin.set_ylabel('Battery SoC (%)', color='#d97706', fontsize=11, fontweight='bold')
ax2_twin.set_ylim(0, 105)
ax2_twin.tick_params(axis='y', labelcolor='#d97706')
ax2_twin.grid(False)

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d %H:%M'))
ax2.set_xlabel('Date / Time (UTC)', fontsize=11, fontweight='bold')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()


## 6. Economic TOU Arbitrage & Tariff Impact Audit

In addition to ensuring grid safety, shifting solar generation from midday to the evening peak monetizes the wide spread in South African electricity tariffs.

Under the **Eskom Megaflex Time-of-Use (TOU)** tariff structure:
- **Peak Hours (17:00–19:00 Winter Weekdays)**: **R 4.95 / kWh**
- **Standard Hours (06:00–09:00, 09:00–17:00 Weekdays)**: **R 1.78 / kWh**
- **Off-Peak Hours (22:00–06:00 Weekdays, all Weekend hours)**: **R 0.92 / kWh**

We calculate energy purchase costs across the feeder with and without BESS dispatch.


In [ ]:
# Define Eskom Megaflex TOU rate lookup function
def get_megaflex_rate(dt):
    # Weekday check
    is_weekday = dt.weekday() < 5
    # Winter rate applies June, July, August (months 6, 7, 8)
    is_winter = dt.month in [6, 7, 8]
    hour = dt.hour
    
    if is_weekday:
        if is_winter:
            if (6 <= hour < 9) or (17 <= hour < 19):
                return 4.95  # Winter Peak
            elif (9 <= hour < 17) or (19 <= hour < 22):
                return 1.78  # Winter Standard
            else:
                return 0.92  # Winter Off-Peak
        else:
            if (7 <= hour < 10) or (18 <= hour < 20):
                return 2.85  # Summer Peak
            elif (6 <= hour < 7) or (10 <= hour < 18) or (20 <= hour < 22):
                return 1.45  # Summer Standard
            else:
                return 0.78  # Summer Off-Peak
    else:
        # Weekends
        if 12 <= hour < 18:
            return 1.35 if is_winter else 1.10
        else:
            return 0.92 if is_winter else 0.78

# Timestamps are in UTC; convert to Durban local time (SAST, UTC+2) for tariff lookup
if timestamps.dt.tz is not None:
    timestamps_sast = timestamps.dt.tz_convert('Africa/Johannesburg')
else:
    timestamps_sast = timestamps.dt.tz_localize('UTC').dt.tz_convert('Africa/Johannesburg')

tariff_rates = np.array([get_megaflex_rate(dt) for dt in timestamps_sast])
interval_hours = 5.0 / 60.0

grid_kwh_unmitigated = np.maximum(feeder_net_unmitigated, 0) * interval_hours
grid_kwh_mitigated = np.maximum(feeder_net_mitigated, 0) * interval_hours

cost_unmitigated = grid_kwh_unmitigated * tariff_rates
cost_mitigated = grid_kwh_mitigated * tariff_rates

total_cost_unmitigated = cost_unmitigated.sum()
total_cost_mitigated = cost_mitigated.sum()
total_savings = total_cost_unmitigated - total_cost_mitigated

print("=== 6-MONTH TOU ECONOMIC ARBITRAGE AUDIT ===")
print(f"Total Electricity Cost (Baseline / No BESS):  R {total_cost_unmitigated:,.2f}")
print(f"Total Electricity Cost (With BESS Dispatch): R {total_cost_mitigated:,.2f}")
print(f"Net Electricity Savings via BESS Arbitrage:  R {total_savings:,.2f} ({100*total_savings/total_cost_unmitigated:.1f}% savings)")

# Monthly Breakdown
monthly_df = pd.DataFrame({
    'timestamp': timestamps_sast.dt.tz_localize(None),
    'cost_baseline': cost_unmitigated,
    'cost_bess': cost_mitigated,
    'savings': cost_unmitigated - cost_mitigated
})
monthly_df['month'] = monthly_df['timestamp'].dt.to_period('M')
monthly_summary = monthly_df.groupby('month')[['cost_baseline', 'cost_bess', 'savings']].sum()

if hasattr(monthly_summary, 'map'):
    display(monthly_summary.map(lambda x: f"R {x:,.0f}"))
else:
    display(monthly_summary.applymap(lambda x: f"R {x:,.0f}"))


## 7. Exporting Standardized Grid Dispatch Telemetry via `pvlib_to_odse`

Finally, we use **`pvlib_to_odse`** to package our modeled feeder net power and dispatch telemetry back into standard ODS-E interchange records. This allows downstream systems—such as SCADA archives, billing reconciliations, or carbon accounting audits—to consume the data without custom ETL scripts.


In [ ]:
# Package modeled feeder telemetry into a pvlib-compliant DataFrame
export_df = pd.DataFrame({
    'p_mp': feeder_net_mitigated.values,
    'ac_energy': (np.maximum(feeder_net_mitigated, 0) * interval_hours).values,
    'poa_global': total_solar.values
}, index=pd.DatetimeIndex(timestamps))

# Convert to standard ODS-E schema payload
feeder_metadata = {
    'substation_id': 'DBN-11KV-NORTH',
    'feeder_id': 'FDR-IND-04',
    'transformer_kva': 5000,
    'mitigation_strategy': 'BESS_COORDINATED_DISPATCH',
    'currency': 'ZAR'
}

odse_export = pvlib_to_odse(
    df=export_df.head(10), # First 10 intervals
    site_id='Durban-North-11kV-Feeder',
    metadata=feeder_metadata
)

print("Standard ODS-E Interchange Export Generated:")
print(f"- Metadata Keys: {list(odse_export['metadata'].keys())}")
print(f"- Telemetry Records Exported: {len(odse_export['telemetry'])}")
print(f"- Sample Exported ODS-E Record:")
print(json.dumps(odse_export['telemetry'][0], indent=2))
print("\n✓ End-to-end ODS-E + pvlib distribution grid modeling workflow complete!")
